# WC Experience

## Objective
World cups are arguably the biggest stage in world sports, with that many things make them unique. Anxiety and moral may be affected by world cup experience, tactics and the essence of the tournament is unique. I want to quantify how team experience (for both managers and players)  m,ay affect performance:
- Manager WC Experience
- Player WC Experience

In this notebook I will explore the relations each of these have with goals scored.

 ## Libraries

In [235]:
library(tidyverse)
library(here)
library(car)
library(stargazer)

## Data

In [236]:
ELO <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "ELOSScores.rds"))
Games <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullTeamGames.rds"))

set.seed(2026)

ELOByTournament <- ELO %>%
    pivot_longer(
        cols = starts_with("WC_"),
        names_to = "tournament_year",
        values_to = "elo"
    ) %>%
    mutate(tournament_id = str_replace(tournament_year, "_", "-")) %>%
    select(team, tournament_id, team_ELO = elo)

OpponentELOByTournament <- ELOByTournament %>%
    rename(opponent_name = team, opponent_ELO = team_ELO)

Games <- Games %>%
    rename(team = team_name) %>%
    left_join(ELOByTournament, by = c("team", "tournament_id")) %>%
    left_join(OpponentELOByTournament, by = c("opponent_name", "tournament_id"))

Games %>%
    slice_sample(n=10)

match_id,tournament_id,stage_name,group_stage,goals_against,goals_for,team_id,opponent_id,city_name,team,opponent_name,match_date,match_time,last_game_date,rest_days,team_ELO,opponent_ELO
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<date>,<dbl>,<int>,<int>
M-2014-51,WC-2014,round of 16,0,2,1,T-46,T-48,Fortaleza,Mexico,Netherlands,2014-06-29,13:00,2014-06-23,6,1796,2003
M-2010-50,WC-2010,round of 16,0,2,1,T-83,T-32,Rustenburg,United States,Ghana,2010-06-26,20:30,2010-06-23,3,1802,1686
M-2022-03,WC-2022,group stage,1,2,0,T-65,T-48,Doha,Senegal,Netherlands,2022-11-21,19:00,NA,7,1702,1929
M-2018-02,WC-2018,group stage,1,1,0,T-26,T-84,Yekaterinburg,Egypt,Uruguay,2018-06-15,17:00,NA,7,1650,1866
M-2010-58,WC-2010,quarter-finals,0,1,1,T-32,T-84,Johannesburg,Ghana,Uruguay,2010-07-02,20:30,2010-06-26,6,1686,1814
M-2010-56,WC-2010,round of 16,0,0,1,T-73,T-58,Cape Town,Spain,Portugal,2010-06-29,20:30,2010-06-25,4,2094,1874
M-2018-01,WC-2018,group stage,1,0,5,T-62,T-63,Moscow,Russia,Saudi Arabia,2018-06-14,18:00,NA,7,1696,1570
M-2010-16,WC-2010,group stage,1,1,0,T-73,T-75,Durban,Spain,Switzerland,2010-06-16,16:00,NA,7,2094,1792
M-2014-49,WC-2014,round of 16,0,1,1,T-13,T-09,Belo Horizonte,Chile,Brazil,2014-06-28,13:00,2014-06-23,5,1914,2132


## Define simulation test function

In [237]:
compare_poisson_models <- function(data, formulas, model_names, outcome = "goals_for",
                                   n_sims = 100, train_prop = 0.8, seed = 2026){

    set.seed(seed)

    log_loss_matrix <- matrix(
        NA,
        nrow = n_sims,
        ncol = length(formulas)
    )

    colnames(log_loss_matrix) <- model_names

    for(i in 1:n_sims){

        train_index <- sample(
            1:nrow(data),
            size = train_prop * nrow(data)
        )

        train_data <- data[train_index, ]
        test_data  <- data[-train_index, ]

        for(j in 1:length(formulas)){

            model <- glm(
                as.formula(formulas[j]),
                family = poisson,
                data = train_data
            )

            pred <- predict(
                model,
                newdata = test_data,
                type = "response"
            )

            log_loss_matrix[i, j] <- -mean(
                dpois(
                    x = test_data[[outcome]],
                    lambda = pred,
                    log = TRUE
                )
            )
        }
    }

    results <- data.frame(
        Model = model_names,
        Mean_Log_Loss = colMeans(log_loss_matrix),
        SD_Log_Loss = apply(log_loss_matrix, 2, sd)
    )

    results <- results[order(results$Mean_Log_Loss), ]

    best_model <- colnames(log_loss_matrix)[apply(log_loss_matrix, 1, which.min)]

    win_table <- data.frame(
        Model = names(table(best_model)),
        Wins = as.numeric(table(best_model))
    )

    return(
        list(
            results = results,
            wins = win_table,
            log_losses = as.data.frame(log_loss_matrix)
        )
    )
}

In [238]:
test_add_one_poisson <- function(data, base_formula, candidate_vars,
                                 outcome = "goals_for",
                                 n_sims = 100,
                                 train_prop = 0.8,
                                 seed = 2026){

    set.seed(seed)

    # Create the same train/test splits for every model
    splits <- vector("list", n_sims)

    for(i in 1:n_sims){
        splits[[i]] <- sample(
            1:nrow(data),
            size = floor(train_prop * nrow(data))
        )
    }

    # Function to calculate log loss for one formula
    get_log_losses <- function(formula_string){

        log_losses <- c()

        for(i in 1:n_sims){

            train_index <- splits[[i]]

            train_data <- data[train_index, ]
            test_data  <- data[-train_index, ]

            model <- glm(
                as.formula(formula_string),
                family = poisson,
                data = train_data
            )

            pred <- predict(
                model,
                newdata = test_data,
                type = "response"
            )

            log_losses[i] <- -mean(
                dpois(
                    x = test_data[[outcome]],
                    lambda = pred,
                    log = TRUE
                )
            )
        }

        return(log_losses)
    }

    # Base model
    base_log_losses <- get_log_losses(base_formula)

    results <- data.frame(
        Variable = "CURRENT BASE",
        Formula = base_formula,
        Mean_Log_Loss = mean(base_log_losses),
        SD_Log_Loss = sd(base_log_losses),
        Improvement_vs_Base = 0,
        Wins_vs_Base = NA
    )

    # Candidate variables
    for(var in candidate_vars){

        formula_string <- paste(
            base_formula,
            "+",
            var
        )

        log_losses <- get_log_losses(formula_string)

        results <- rbind(
            results,
            data.frame(
                Variable = var,
                Formula = formula_string,
                Mean_Log_Loss = mean(log_losses),
                SD_Log_Loss = sd(log_losses),
                Improvement_vs_Base = mean(base_log_losses) - mean(log_losses),
                Wins_vs_Base = sum(log_losses < base_log_losses)
            )
        )
    }

    results <- results[order(results$Mean_Log_Loss), ]

    return(results)
}

In [239]:
forward_select_poisson <- function(data, base_formula, candidate_vars,
                                   outcome = "goals_for",
                                   n_sims = 100,
                                   train_prop = 0.8,
                                   seed = 2026,
                                   min_improvement = 0){

    set.seed(seed)

    # Create fixed train/test splits
    splits <- vector("list", n_sims)

    for(i in 1:n_sims){
        splits[[i]] <- sample(
            1:nrow(data),
            size = floor(train_prop * nrow(data))
        )
    }

    # Function to calculate log losses for one formula
    get_log_losses <- function(formula_string){

        log_losses <- c()

        for(i in 1:n_sims){

            train_index <- splits[[i]]

            train_data <- data[train_index, ]
            test_data  <- data[-train_index, ]

            model <- glm(
                as.formula(formula_string),
                family = poisson,
                data = train_data
            )

            pred <- predict(
                model,
                newdata = test_data,
                type = "response"
            )

            log_losses[i] <- -mean(
                dpois(
                    x = test_data[[outcome]],
                    lambda = pred,
                    log = TRUE
                )
            )
        }

        return(log_losses)
    }

    selected_vars <- c()
    remaining_vars <- candidate_vars
    current_formula <- base_formula
    current_log_losses <- get_log_losses(current_formula)
    current_mean_log_loss <- mean(current_log_losses)

    selection_history <- data.frame(
        Step = 0,
        Added_Variable = "BASE MODEL",
        Formula = current_formula,
        Mean_Log_Loss = current_mean_log_loss,
        Improvement = 0,
        Wins_vs_Previous = NA
    )

    step <- 1

    while(length(remaining_vars) > 0){

        step_results <- data.frame()

        for(var in remaining_vars){

            test_formula <- paste(
                current_formula,
                "+",
                var
            )

            test_log_losses <- get_log_losses(test_formula)

            step_results <- rbind(
                step_results,
                data.frame(
                    Variable = var,
                    Formula = test_formula,
                    Mean_Log_Loss = mean(test_log_losses),
                    Improvement = current_mean_log_loss - mean(test_log_losses),
                    Wins_vs_Current = sum(test_log_losses < current_log_losses)
                )
            )
        }

        step_results <- step_results[order(step_results$Mean_Log_Loss), ]

        best_var <- step_results$Variable[1]
        best_formula <- step_results$Formula[1]
        best_mean_log_loss <- step_results$Mean_Log_Loss[1]
        best_improvement <- step_results$Improvement[1]
        best_wins <- step_results$Wins_vs_Current[1]

        if(best_improvement <= min_improvement){
            break
        }

        selected_vars <- c(selected_vars, best_var)
        remaining_vars <- setdiff(remaining_vars, best_var)

        current_formula <- best_formula
        current_log_losses <- get_log_losses(current_formula)
        current_mean_log_loss <- best_mean_log_loss

        selection_history <- rbind(
            selection_history,
            data.frame(
                Step = step,
                Added_Variable = best_var,
                Formula = current_formula,
                Mean_Log_Loss = current_mean_log_loss,
                Improvement = best_improvement,
                Wins_vs_Previous = best_wins
            )
        )

        step <- step + 1
    }

    return(
        list(
            selected_vars = selected_vars,
            final_formula = current_formula,
            selection_history = selection_history,
            remaining_vars = remaining_vars
        )
    )
}

## Manager Experience

### Join manager experience both for opponent and teams

In [240]:
managerwchist <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "ManagerWC_history.rds"))

Games <- left_join(Games,managerwchist, by = c("tournament_id", "team_id"))

managerwchist_opp <- managerwchist %>%
    rename(
        opponent_id = team_id,
        opp_team_name = team_name,
        opp_team_code = team_code,
        opp_manager_id = manager_id,
        opp_coach_name_nft = coach_name_nft,
        opp_coach_url_nft = coach_url_nft,
        opp_pre_wc_team_matches_coached = pre_wc_team_matches_coached,
        opp_coach_team_matches_source = coach_team_matches_source,
        opp_coach_team_matches_cutoff_date = coach_team_matches_cutoff_date,
        opp_prior_wc_matches_coached = prior_wc_matches_coached,
        opp_prior_wc_group_matches_coached = prior_wc_group_matches_coached,
        opp_prior_wc_round_of_16_matches_coached = prior_wc_round_of_16_matches_coached,
        opp_prior_wc_quarterfinal_matches_coached = prior_wc_quarterfinal_matches_coached,
        opp_prior_wc_semifinal_matches_coached = prior_wc_semifinal_matches_coached,
        opp_prior_wc_third_place_matches_coached = prior_wc_third_place_matches_coached,
        opp_prior_wc_final_matches_coached = prior_wc_final_matches_coached,
        opp_prior_world_cups_coached = prior_world_cups_coached
    )

Games <- Games %>%
    left_join(
        managerwchist_opp,
        by = c("tournament_id", "opponent_id")
    )

set.seed(2026)
Games%>%
    slice_sample(n=10)

match_id,tournament_id,stage_name,group_stage,goals_against,goals_for,team_id,opponent_id,city_name,team,...,opp_coach_team_matches_source,opp_coach_team_matches_cutoff_date,opp_prior_wc_matches_coached,opp_prior_wc_group_matches_coached,opp_prior_wc_round_of_16_matches_coached,opp_prior_wc_quarterfinal_matches_coached,opp_prior_wc_semifinal_matches_coached,opp_prior_wc_third_place_matches_coached,opp_prior_wc_final_matches_coached,opp_prior_world_cups_coached
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,...,<chr>,<date>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
M-2014-51,WC-2014,round of 16,0,2,1,T-46,T-48,Fortaleza,Mexico,...,National Football Teams coach match logs,2014-06-12,0,0,0,0,0,0,0,0
M-2010-50,WC-2010,round of 16,0,2,1,T-83,T-32,Rustenburg,United States,...,National Football Teams coach match logs,2010-06-11,0,0,0,0,0,0,0,0
M-2022-03,WC-2022,group stage,1,2,0,T-65,T-48,Doha,Senegal,...,National Football Teams coach match logs,2022-11-20,7,3,1,1,1,1,0,1
M-2018-02,WC-2018,group stage,1,1,0,T-26,T-84,Yekaterinburg,Egypt,...,National Football Teams coach match logs,2018-06-14,15,9,3,1,1,1,0,3
M-2010-58,WC-2010,quarter-finals,0,1,1,T-32,T-84,Johannesburg,Ghana,...,National Football Teams coach match logs,2010-06-11,4,3,1,0,0,0,0,1
M-2010-56,WC-2010,round of 16,0,0,1,T-73,T-58,Cape Town,Spain,...,National Football Teams coach match logs,2010-06-11,0,0,0,0,0,0,0,0
M-2018-01,WC-2018,group stage,1,0,5,T-62,T-63,Moscow,Russia,...,National Football Teams coach match logs,2018-06-14,0,0,0,0,0,0,0,0
M-2010-16,WC-2010,group stage,1,1,0,T-73,T-75,Durban,Spain,...,National Football Teams coach match logs,2010-06-11,0,0,0,0,0,0,0,0
M-2014-49,WC-2014,round of 16,0,1,1,T-13,T-09,Belo Horizonte,Chile,...,National Football Teams coach match logs,2014-06-12,14,6,2,2,2,1,1,2


### Test relation betwene team coach wc experience and goals for, accouinting for ELO

I'll drop the brazil 7-1 game because its an extreme outlier with a finalist wc coach which might affect the smaller sample

Let's compare the ELO only model to models with. our coach wc experience variables

In [241]:
Games <- Games %>%
    filter(match_id!="M-2014-61")

In [242]:
base_formula <- "goals_for ~ I(team_ELO - opponent_ELO)"

candidate_vars <- c(
    "prior_wc_matches_coached",
    "prior_wc_group_matches_coached",
    "prior_wc_round_of_16_matches_coached",
    "prior_wc_quarterfinal_matches_coached",
    "prior_wc_semifinal_matches_coached",
    "prior_wc_third_place_matches_coached",
    "prior_wc_final_matches_coached",
    "prior_world_cups_coached"
)

test_add_one_poisson(
    data = Games,
    base_formula = base_formula,
    candidate_vars = candidate_vars
)

,Variable,Formula,Mean_Log_Loss,SD_Log_Loss,Improvement_vs_Base,Wins_vs_Base
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>
1,CURRENT BASE,goals_for ~ I(team_ELO - opponent_ELO),1.421375,0.06540084,0.000000000,NA
8,prior_wc_final_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + prior_wc_final_matches_coached,1.422554,0.06474448,-0.001179603,52
6,prior_wc_semifinal_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + prior_wc_semifinal_matches_coached,1.423560,0.06570574,-0.002185129,52
3,prior_wc_group_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + prior_wc_group_matches_coached,1.423824,0.06531466,-0.002449023,20
9,prior_world_cups_coached,goals_for ~ I(team_ELO - opponent_ELO) + prior_world_cups_coached,1.423824,0.06531466,-0.002449023,20
2,prior_wc_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + prior_wc_matches_coached,1.423979,0.06542463,-0.002604449,30
5,prior_wc_quarterfinal_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + prior_wc_quarterfinal_matches_coached,1.424117,0.06577676,-0.002742220,7
4,prior_wc_round_of_16_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + prior_wc_round_of_16_matches_coached,1.424131,0.06575944,-0.002755967,17
7,prior_wc_third_place_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + prior_wc_third_place_matches_coached,1.424310,0.06597189,-0.002934936,22


This variables dont's seem to better our model. Lets test the opponent coach variables now.

In [243]:
base_formula <- "goals_for ~ I(team_ELO - opponent_ELO)"

candidate_vars <- c(
    "opp_prior_wc_matches_coached",
    "opp_prior_wc_group_matches_coached",
    "opp_prior_wc_round_of_16_matches_coached",
    "opp_prior_wc_quarterfinal_matches_coached",
    "opp_prior_wc_semifinal_matches_coached",
    "opp_prior_wc_third_place_matches_coached",
    "opp_prior_wc_final_matches_coached",
    "opp_prior_world_cups_coached"
)

test_add_one_poisson(
    data = Games,
    base_formula = base_formula,
    candidate_vars = candidate_vars
)

,Variable,Formula,Mean_Log_Loss,SD_Log_Loss,Improvement_vs_Base,Wins_vs_Base
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>
3,opp_prior_wc_group_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_group_matches_coached,1.417799,0.06534916,0.0035762076,78
9,opp_prior_world_cups_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached,1.417799,0.06534916,0.0035762076,78
2,opp_prior_wc_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_matches_coached,1.418802,0.06589304,0.0025725320,74
4,opp_prior_wc_round_of_16_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_round_of_16_matches_coached,1.420184,0.06650113,0.0011905946,68
8,opp_prior_wc_final_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_final_matches_coached,1.421210,0.06601798,0.0001650924,60
1,CURRENT BASE,goals_for ~ I(team_ELO - opponent_ELO),1.421375,0.06540084,0.0000000000,NA
6,opp_prior_wc_semifinal_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_semifinal_matches_coached,1.422096,0.06579889,-0.0007211502,54
5,opp_prior_wc_quarterfinal_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_quarterfinal_matches_coached,1.422903,0.06548143,-0.0015280237,24
7,opp_prior_wc_third_place_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_third_place_matches_coached,1.423284,0.06491613,-0.0019094509,23


Multiple variables seem to potentially better our predictive ability beyond ELO. let's trun a forward selection method to determine which ones to keep.

In [244]:
candidate_vars <- c(
    "opp_prior_wc_matches_coached",
    "opp_prior_wc_round_of_16_matches_coached",
    "opp_prior_wc_semifinal_matches_coached",
    "opp_prior_world_cups_coached"
)

base_formula <- paste(
    "goals_for ~ I(team_ELO - opponent_ELO)",
    "+ opp_prior_wc_final_matches_coached"
)

forward_step_2 <- test_add_one_poisson(
    data = Games,
    base_formula = base_formula,
    candidate_vars = candidate_vars
)

forward_step_2

,Variable,Formula,Mean_Log_Loss,SD_Log_Loss,Improvement_vs_Base,Wins_vs_Base
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>
5,opp_prior_world_cups_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_final_matches_coached + opp_prior_world_cups_coached,1.420387,0.06600522,0.0008227474,72
1,CURRENT BASE,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_final_matches_coached,1.421210,0.06601798,0.0000000000,NA
2,opp_prior_wc_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_final_matches_coached + opp_prior_wc_matches_coached,1.421385,0.06641291,-0.0001753734,66
3,opp_prior_wc_round_of_16_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_final_matches_coached + opp_prior_wc_round_of_16_matches_coached,1.421962,0.06701297,-0.0007525985,59
4,opp_prior_wc_semifinal_matches_coached,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_final_matches_coached + opp_prior_wc_semifinal_matches_coached,1.423061,0.06542925,-0.0018510236,39


The results suggest prior WCs coached and Prior WC finals coached might be helpful.

Lets run a simulation to compare the ELO only model vs the ELO + wcfinalscoached vs  ELO + wcfinalscoached + wccoached

In [245]:
model_test <- compare_poisson_models(
    data = Games,
    formulas = c(
        "goals_for ~ I(team_ELO - opponent_ELO)",
        "goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_final_matches_coached",
        "goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached",
        "goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_wc_final_matches_coached + opp_prior_world_cups_coached"
    ),
    model_names = c(
        "ELO only",
        "ELO + Opp WC Finals",
        "ELO + OPP WCs Coached",
        "ELO + Opp WC Finals + Opp WCs Coached"
    )
)

model_test$results
model_test$wins

,Model,Mean_Log_Loss,SD_Log_Loss
,<chr>,<dbl>,<dbl>
ELO + OPP WCs Coached,ELO + OPP WCs Coached,1.417799,0.06534916
ELO + Opp WC Finals + Opp WCs Coached,ELO + Opp WC Finals + Opp WCs Coached,1.420387,0.06600522
ELO + Opp WC Finals,ELO + Opp WC Finals,1.421210,0.06601798
ELO only,ELO only,1.421375,0.06540084


Model,Wins
<chr>,<dbl>
ELO + OPP WCs Coached,46
ELO + Opp WC Finals,20
ELO + Opp WC Finals + Opp WCs Coached,25
ELO only,9


The ELO + OPP WCs Coached seems to be our best option here. I want to compare Std. Errors to bette runderstand what happened here.

In [246]:
m1 <- glm(goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_prior_wc_final_matches_coached, family=poisson, data=Games)
m2 <- glm(goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached, family=poisson, data=Games)

stargazer( m1,m2,type = "text")


                                       Dependent variable:     
                                   ----------------------------
                                            goals_for          
                                        (1)            (2)     
---------------------------------------------------------------
I(team_ELO - opponent_ELO)            0.002***      0.001***   
                                      (0.0002)      (0.0002)   
                                                               
opp_prior_world_cups_coached           0.067         0.086**   
                                      (0.048)        (0.042)   
                                                               
opp_prior_wc_final_matches_coached     0.145                   
                                      (0.165)                  
                                                               
Constant                              0.144***      0.146***   
                                      (

<div style="
    border: 3px solid #28a745;
    border-radius: 10px;
    padding: 15px 20px;
    margin: 20px 0;
    background-color: #f8fff9;
    color: #000000;
">

<h3 style="
    margin-top:0;
    color:#000000;
">
📊 Math Check: Multicollinearity
</h3>

<p>
Before moving forward, it is important to pause and discuss a key statistical issue:
<strong>multicollinearity</strong>.
</p>

<p>
In the regression output, look at the numbers in parentheses below each estimate. These numbers are the <strong>standard errors</strong>. For example, if a coefficient estimate is smaller than its standard error, such as an estimate of 0.12 with a standard error of 0.165, that suggests the estimate is relatively unstable.
</p>

<p>
This does not necessarily mean the variable is unimportant. Instead, it means the model is having difficulty determining the precise effect of that variable while simultaneously accounting for the effects of the other predictors.
</p>

<p>
In this case, the issue likely arises because the number of World Cups a manager has coached and the number of World Cup finals they have coached are highly related. Managers who have reached multiple World Cup finals are also likely to have accumulated substantial World Cup experience overall.
</p>

<p>
This phenomenon is known as <strong>multicollinearity</strong>. It occurs when two or more explanatory variables contain very similar information.
</p>

<p>
The main consequence is that it becomes more difficult to determine how much of the observed effect should be attributed to each individual variable. The model may still make accurate predictions, but the interpretation of the coefficients becomes less reliable. That is why we see the log loss for the model is still pretty good while the p-value of the variable is comparatively high.
</p>

<p style="margin-bottom:0;">
<strong>Key takeaway:</strong> Multicollinearity is primarily an interpretation problem rather than a prediction problem. A model can still perform well out of sample even when individual coefficient estimates are unstable.
</p>

</div>

In [247]:
summary( glm(goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached , family=poisson, data=Games))


Call:
glm(formula = goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached, 
    family = poisson, data = Games)

Coefficients:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                  0.1457497  0.0477178   3.054  0.00226 ** 
I(team_ELO - opponent_ELO)   0.0014959  0.0001869   8.003 1.22e-15 ***
opp_prior_world_cups_coached 0.0858766  0.0424656   2.022  0.04315 *  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 648.53  on 509  degrees of freedom
Residual deviance: 581.56  on 507  degrees of freedom
AIC: 1449.9

Number of Fisher Scoring iterations: 5


As we can see the coefficient is positive, which means the most world cups a manager has coached the most gaols we'd expect his rival to score. Although this sounds counterintuitive we need to remember that we've already accounted for team strenght so the effct of a more experienced coach being in a better team is already taken into account. There are many reasons we can hypothetise why this could happen. For example:
- A coach has been in international football for oto long and opponents have learned how to play agianst his style.
- New coaches are more precatious and prefer defensive styles.
- Coaches are more motivated and illusioned in theirfirst world cup and develop more efficient tactis.
Anyways we can not determine the exact reason for this relationship with the current data but we can asses that there is a statistically significant relation.

**The model has evolved from goals_for ~ ELO_difference to goals_for ~ ELO_difference + opp_prior_world_cups_coached.**

## Player Experience

In [248]:
PlayerExperience <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "PlayerExperience.rds"))

Games <- left_join(Games, PlayerExperience, by = c("tournament_id", "team_id"))

opp_PlayerExperience <- PlayerExperience %>%
    rename(
        opponent_id = team_id,
        opp_team_name = team_name,
        opp_team_code = team_code,
        opp_squad_players = squad_players,
        opp_players_with_prior_wc = players_with_prior_wc,
        opp_share_players_with_prior_wc = share_players_with_prior_wc,
        opp_total_prior_wc_matches_played = total_prior_wc_matches_played,
        opp_mean_prior_wc_matches_played = mean_prior_wc_matches_played,
        opp_max_prior_wc_matches_played = max_prior_wc_matches_played,
        opp_total_prior_wc_starts = total_prior_wc_starts,
        opp_total_prior_wc_substitute_appearances = total_prior_wc_substitute_appearances,
        opp_total_prior_wc_group_matches_played = total_prior_wc_group_matches_played,
        opp_total_prior_wc_round_of_16_matches_played = total_prior_wc_round_of_16_matches_played,
        opp_total_prior_wc_quarterfinal_matches_played = total_prior_wc_quarterfinal_matches_played,
        opp_total_prior_wc_semifinal_matches_played = total_prior_wc_semifinal_matches_played,
        opp_total_prior_wc_third_place_matches_played = total_prior_wc_third_place_matches_played,
        opp_total_prior_wc_final_matches_played = total_prior_wc_final_matches_played,
        opp_total_prior_world_cups_played = total_prior_world_cups_played,
        opp_mean_prior_world_cups_played = mean_prior_world_cups_played,
        opp_max_prior_world_cups_played = max_prior_world_cups_played,
        opp_players_with_prior_wc_final = players_with_prior_wc_final,
        opp_players_with_multiple_prior_wcs = players_with_multiple_prior_wcs,
        opp_avg_age = avg_age 
        
    )


Games <- Games %>%
    left_join(
        opp_PlayerExperience,
        by = c("tournament_id", "opponent_id")
    )

In [249]:
base_formula <- "goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached"

candidate_vars <- c(
    "players_with_prior_wc",
    "share_players_with_prior_wc",
    "total_prior_wc_matches_played",
    "mean_prior_wc_matches_played",
    "max_prior_wc_matches_played",
    "total_prior_wc_starts",
    "total_prior_wc_substitute_appearances",
    "total_prior_wc_group_matches_played",
    "total_prior_wc_round_of_16_matches_played",
    "total_prior_wc_quarterfinal_matches_played",
    "total_prior_wc_semifinal_matches_played",
    "total_prior_wc_third_place_matches_played",
    "total_prior_wc_final_matches_played",
    "total_prior_world_cups_played",
    "mean_prior_world_cups_played",
    "max_prior_world_cups_played",
    "players_with_prior_wc_final",
    "players_with_multiple_prior_wcs"
)

test_add_one_poisson(
    data = Games,
    base_formula = base_formula,
    candidate_vars = candidate_vars
)

,Variable,Formula,Mean_Log_Loss,SD_Log_Loss,Improvement_vs_Base,Wins_vs_Base
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>
1,CURRENT BASE,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached,1.417799,0.06534916,0.000000000,NA
3,share_players_with_prior_wc,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + share_players_with_prior_wc,1.418971,0.06504981,-0.001171915,56
2,players_with_prior_wc,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + players_with_prior_wc,1.418987,0.06508694,-0.001188274,59
17,max_prior_world_cups_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + max_prior_world_cups_played,1.419455,0.06556079,-0.001656193,55
6,max_prior_wc_matches_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + max_prior_wc_matches_played,1.419629,0.06507424,-0.001830483,53
9,total_prior_wc_group_matches_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + total_prior_wc_group_matches_played,1.419763,0.06498661,-0.001964339,38
14,total_prior_wc_final_matches_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + total_prior_wc_final_matches_played,1.419955,0.06518700,-0.002156318,30
18,players_with_prior_wc_final,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + players_with_prior_wc_final,1.419955,0.06518700,-0.002156318,30
16,mean_prior_world_cups_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + mean_prior_world_cups_played,1.420024,0.06485423,-0.002225536,8


None of our variables seem to better the model. Let's now try it for opponent player experience.

In [250]:
base_formula <- "goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached"

candidate_vars <- c(
    "opp_players_with_prior_wc",
    "opp_share_players_with_prior_wc",
    "opp_total_prior_wc_matches_played",
    "opp_mean_prior_wc_matches_played",
    "opp_max_prior_wc_matches_played",
    "opp_total_prior_wc_starts",
    "opp_total_prior_wc_substitute_appearances",
    "opp_total_prior_wc_group_matches_played",
    "opp_total_prior_wc_round_of_16_matches_played",
    "opp_total_prior_wc_quarterfinal_matches_played",
    "opp_total_prior_wc_semifinal_matches_played",
    "opp_total_prior_wc_third_place_matches_played",
    "opp_total_prior_wc_final_matches_played",
    "opp_total_prior_world_cups_played",
    "opp_mean_prior_world_cups_played",
    "opp_max_prior_world_cups_played",
    "opp_players_with_prior_wc_final",
    "opp_players_with_multiple_prior_wcs"
)

test_add_one_poisson(
    data = Games,
    base_formula = base_formula,
    candidate_vars = candidate_vars
)

,Variable,Formula,Mean_Log_Loss,SD_Log_Loss,Improvement_vs_Base,Wins_vs_Base
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>
8,opp_total_prior_wc_substitute_appearances,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_total_prior_wc_substitute_appearances,1.415614,0.06446519,0.0021850231,66
14,opp_total_prior_wc_final_matches_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_total_prior_wc_final_matches_played,1.415785,0.06651208,0.0020140275,60
18,opp_players_with_prior_wc_final,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_players_with_prior_wc_final,1.415785,0.06651208,0.0020140275,60
19,opp_players_with_multiple_prior_wcs,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs,1.416315,0.06349546,0.0014836663,71
1,CURRENT BASE,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached,1.417799,0.06534916,0.0000000000,NA
9,opp_total_prior_wc_group_matches_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_total_prior_wc_group_matches_played,1.417929,0.06485328,-0.0001304174,67
15,opp_total_prior_world_cups_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_total_prior_world_cups_played,1.418142,0.06479734,-0.0003433607,66
16,opp_mean_prior_world_cups_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_mean_prior_world_cups_played,1.418182,0.06483380,-0.0003837807,65
4,opp_total_prior_wc_matches_played,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_total_prior_wc_matches_played,1.418749,0.06500733,-0.0009500914,58


In [251]:
candidate_vars <- c(
    "opp_players_with_prior_wc",
    "opp_share_players_with_prior_wc",
    "opp_total_prior_wc_matches_played",
    "opp_mean_prior_wc_matches_played",
    "opp_max_prior_wc_matches_played",
    "opp_total_prior_wc_starts",
    "opp_total_prior_wc_substitute_appearances",
    "opp_total_prior_wc_group_matches_played",
    "opp_total_prior_wc_round_of_16_matches_played",
    "opp_total_prior_wc_quarterfinal_matches_played",
    "opp_total_prior_wc_semifinal_matches_played",
    "opp_total_prior_wc_third_place_matches_played",
    "opp_total_prior_wc_final_matches_played",
    "opp_total_prior_world_cups_played",
    "opp_mean_prior_world_cups_played",
    "opp_max_prior_world_cups_played",
    "opp_players_with_prior_wc_final",
    "opp_players_with_multiple_prior_wcs"
)

forward_opp_players <- forward_select_poisson(
    data = Games,
    base_formula = "goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached",
    candidate_vars = candidate_vars,
    min_improvement = 0
)

forward_opp_players$selection_history
forward_opp_players$final_formula
forward_opp_players$selected_vars

Step,Added_Variable,Formula,Mean_Log_Loss,Improvement,Wins_vs_Previous
<dbl>,<chr>,<chr>,<dbl>,<dbl>,<int>
0,BASE MODEL,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached,1.417799,0.000000000,NA
1,opp_total_prior_wc_substitute_appearances,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_total_prior_wc_substitute_appearances,1.415614,0.002185023,66


[1] "goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_total_prior_wc_substitute_appearances"

[1] "opp_total_prior_wc_substitute_appearances"

Log loss suggest we should add opp_total_prior_wc_substitute_appearances, and this might be a real trend. Nonetheless I don't feel comfortable using this variable for multiple reason:
- Substitution rules changed during covid allowing more subs, 2022 was teh first world cup with this rule so we have no training data on how that might affect this.
- Subs are a way lower number than total players and the sample size becomes smaller.
- It only beats the baseline model 60/100 times which sugest the trend might not be consistent across all data.
- WC appearences per tournament beats the baseline 71/100 times. Has a bigger sample size. And makes more sense logically.

In [252]:
m1 <- glm(goals_for ~ I(team_ELO - opponent_ELO) +opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs,family = poisson,data = Games)
m2 <- glm(goals_for ~ I(team_ELO - opponent_ELO) +opp_prior_world_cups_coached + opp_total_prior_wc_substitute_appearances,family = poisson,data = Games)

stargazer(
    m1, m2,
    type = "text",
    report = "vcp"
)


                                              Dependent variable:     
                                          ----------------------------
                                                   goals_for          
                                               (1)            (2)     
----------------------------------------------------------------------
I(team_ELO - opponent_ELO)                    0.002          0.002    
                                            p = 0.000      p = 0.000  
                                                                      
opp_prior_world_cups_coached                  0.076          0.080    
                                            p = 0.080      p = 0.065  
                                                                      
opp_players_with_multiple_prior_wcs           0.039                   
                                            p = 0.053                 
                                                                      
opp_t

It is interesting to observe that more wc experience by players results in more projected goals for the opponent. This might very well be a causal relation but to me it potentially sound like a proxy to age. More WCs played -> older players. Lets account for avg_Age.

In [253]:
m1 <- glm(goals_for ~ I(team_ELO - opponent_ELO) +opp_prior_world_cups_coached + opp_avg_age + opp_players_with_multiple_prior_wcs,family = poisson,data = Games)
m2 <- glm(goals_for ~ I(team_ELO - opponent_ELO) +opp_prior_world_cups_coached + opp_avg_age + opp_total_prior_wc_substitute_appearances,family = poisson,data = Games)

stargazer(
    m1, m2,
    type = "text",
    report = "vcp"
)


                                              Dependent variable:     
                                          ----------------------------
                                                   goals_for          
                                               (1)            (2)     
----------------------------------------------------------------------
I(team_ELO - opponent_ELO)                    0.002          0.002    
                                            p = 0.000      p = 0.000  
                                                                      
opp_prior_world_cups_coached                  0.070          0.074    
                                            p = 0.107      p = 0.086  
                                                                      
opp_avg_age                                   0.051          0.045    
                                            p = 0.196      p = 0.269  
                                                                      
opp_p

We can see this lowers the effect of both variables on goals_for. More interestingly it makes opp_total_prior_wc_substitute_appearances  not significant at a 10% significance level but opp_players_with_multiple_prior_wcs remains significant. We can see a dramatic increase in p for opp_total_prior_wc_substitute_appearances which suggest that it was in fact mostly a proxy for age. On the other hand opp_players_with_multiple_prior_wcs p-value  increases much less dramtically suggesting that even if some of the effect was attributed to age there's still some valuable insight brought from the variable beyond Age.

### Player Age

I will continue for now with goals_for ~ ELO_difference + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs .

Lets test Age both for opponents and teams.

In [254]:
candidate_vars <- c(
    "opp_avg_age",
    "avg_age",
    "poly(opp_avg_age,2)",
    "poly(avg_age,2)",
    "I(opp_avg_age^2)",
    "I(avg_age^2)"
)

forward_opp_players <- forward_select_poisson(
    data = Games,
    base_formula = "goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs",
    candidate_vars = candidate_vars,
    min_improvement = 0
)

forward_opp_players$selection_history

Step,Added_Variable,Formula,Mean_Log_Loss,Improvement,Wins_vs_Previous
<dbl>,<chr>,<chr>,<dbl>,<dbl>,<int>
0,BASE MODEL,goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs,1.416315,0.0000000000,NA
1,I(avg_age^2),goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs + I(avg_age^2),1.411288,0.0050271760,66
2,"poly(opp_avg_age,2)","goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs + I(avg_age^2) + poly(opp_avg_age,2)",1.410453,0.0008352034,66


This suggest both opponent and team average age provide insight beyond ELO, prior world cups by coach, and polayers with multiple wcs.

In [255]:
m1 <- glm(goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs+I(avg_age**2)+opp_avg_age+I(opp_avg_age**2), family=poisson, Games)
m2 <- glm(goals_for ~ I(team_ELO - opponent_ELO) + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs+avg_age+opp_avg_age, family=poisson, Games)

stargazer(
    m1,
    m2,
    type = "html",
    out = "models.html",
    report="vcp"
)


<table style="text-align:center"><tr><td colspan="3" style="border-bottom: 1px solid black"></td></tr><tr><td style="text-align:left"></td><td colspan="2"><em>Dependent variable:</em></td></tr>
<tr><td></td><td colspan="2" style="border-bottom: 1px solid black"></td></tr>
<tr><td style="text-align:left"></td><td colspan="2">goals_for</td></tr>
<tr><td style="text-align:left"></td><td>(1)</td><td>(2)</td></tr>
<tr><td colspan="3" style="border-bottom: 1px solid black"></td></tr><tr><td style="text-align:left">I(team_ELO - opponent_ELO)</td><td>0.002</td><td>0.002</td></tr>
<tr><td style="text-align:left"></td><td>p = 0.000</td><td>p = 0.000</td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">opp_prior_world_cups_coached</td><td>0.078</td><td>0.069</td></tr>
<tr><td style="text-align:left"></td><td>p = 0.076</td><td>p = 0.112</td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">opp_playe


<table style="text-align:center"><tr><td colspan="3" style="border-bottom: 1px solid black"></td></tr><tr><td style="text-align:left"></td><td colspan="2"><em>Dependent variable:</em></td></tr>
<tr><td></td><td colspan="2" style="border-bottom: 1px solid black"></td></tr>
<tr><td style="text-align:left"></td><td colspan="2">goals_for</td></tr>
<tr><td style="text-align:left"></td><td>(1)</td><td>(2)</td></tr>
<tr><td colspan="3" style="border-bottom: 1px solid black"></td></tr><tr><td style="text-align:left">I(team_ELO - opponent_ELO)</td><td>0.002</td><td>0.002</td></tr>
<tr><td style="text-align:left"></td><td>p = 0.000</td><td>p = 0.000</td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">opp_prior_world_cups_coached</td><td>0.078</td><td>0.069</td></tr>
<tr><td style="text-align:left"></td><td>p = 0.076</td><td>p = 0.112</td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">opp_players_with_multiple_prior_wcs</td><td>0.037</td><td>0.038</td></tr>
<tr><td style="text-align:left"></td><td>p = 0.074</td><td>p = 0.063</td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">I(avg_age2)</td><td>-0.002</td><td></td></tr>
<tr><td style="text-align:left"></td><td>p = 0.007</td><td></td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">avg_age</td><td></td><td>-0.106</td></tr>
<tr><td style="text-align:left"></td><td></td><td>p = 0.006</td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">opp_avg_age</td><td>-2.504</td><td>0.044</td></tr>
<tr><td style="text-align:left"></td><td>p = 0.102</td><td>p = 0.267</td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">I(opp_avg_age2)</td><td>0.047</td><td></td></tr>
<tr><td style="text-align:left"></td><td>p = 0.096</td><td></td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td style="text-align:left">Constant</td><td>35.076</td><td>1.783</td></tr>
<tr><td style="text-align:left"></td><td>p = 0.093</td><td>p = 0.249</td></tr>
<tr><td style="text-align:left"></td><td></td><td></td></tr>
<tr><td colspan="3" style="border-bottom: 1px solid black"></td></tr><tr><td style="text-align:left">Observations</td><td>510</td><td>510</td></tr>
<tr><td style="text-align:left">Log Likelihood</td><td>-714.105</td><td>-715.474</td></tr>
<tr><td style="text-align:left">Akaike Inf. Crit.</td><td>1,442.211</td><td>1,442.947</td></tr>
<tr><td colspan="3" style="border-bottom: 1px solid black"></td></tr><tr><td style="text-align:left"><em>Note:</em></td><td colspan="2" style="text-align:right"><sup>*</sup>p<0.1; <sup>**</sup>p<0.05; <sup>***</sup>p<0.01</td></tr>
</table>


In [256]:
BIC(m1)
BIC(m2)

[1] 1471.852

[1] 1468.354

Although BIC goes down I'm not too concerned about complexity here. We've been testing the model on out of sample data and the added complexity seems to be justified by strong evidence that the relationship between opp_avg_age and goals_for is quadratic in nature (p=0.267 for non-quadratic term by itself).

### - Final working model by the end of this notebook: **goals_for ~ ELO_diff + opp_prior_world_cups_coached + opp_players_with_multiple_prior_wcs+(avg_age^2)+opp_avg_age+(opp_avg_age^2)**
### - BIC = **1471.85**
### - Out of Sample log loss = **1.410453**

Save Games dataset in RDS to make sure we're using the same dataset in future modeling

In [258]:
saveRDS(Games, here("1.DataCleaning-R", "Data", "RDS", "GamesModeling.rds"))